In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

In [3]:
batch_size = 5
sequence_length = 15
input_size = 10
hidden_size = 20
num_layers = 2

In [4]:
lstm = nn.LSTM(input_size = input_size,
hidden_size=hidden_size,
num_layers=num_layers,
batch_first = True)

rand=torch.randn(batch_size, sequence_length, input_size)

h0=torch.zeros(num_layers, batch_size, hidden_size)
c0=torch.zeros(num_layers, batch_size, hidden_size)

method_1_outs, (hn,cn) = lstm(rand, (h0, c0))

h = torch.zeros(num_layers, batch_size, hidden_size)
c = torch.zeros(num_layers, batch_size, hidden_size)

outs = []

for i in range(sequence_length):
    token = rand[:, i, :].unsqueeze(1)
    out, (h,c) = lstm(token, (h,c))

    outs.append(out)

method_2_outs = torch.concat(outs, axis=1)

torch.allclose(method_1_outs, method_2_outs)

True

In [14]:
path_to_data = "/home/zhalas/Downloads/data/harry_potter._txt"
text_files = os.listdir(path_to_data)

all_text = ""
for book in text_files:
    path_to_book = os.path.join(path_to_data, book)

    with open(path_to_book, "r") as f:
        text = f.readlines()

    text = [line for line in text if "Page" not in line]
    text = " ".join(text).replace("\n", "")
    text = [word for word in text.split(" ") if len(word) > 0]
    text = " ".join(text)
    all_text+=text

In [15]:
unique_chars = sorted(list(set(all_text)))

char2idx = {c:i for (i,c) in enumerate(unique_chars)}
idx2char = {i:c for (i,c) in enumerate(unique_chars)}

In [16]:
class DataBuilder:
    def __init__(self, seq_len=300, text=all_text):

        self.seq_len = seq_len
        self.text = text
        self.file_length = len(text)

    def grab_random_sample(self):

        start = np.random.randint(0, self.file_length-self.seq_len)
        end = start + self.seq_len
        text_slice = self.text[start:end]

        input_text = text_slice[:-1]
        label = text_slice[1:]

        input_text = torch.tensor([char2idx[c] for c in input_text])
        label = torch.tensor([char2idx[c] for c in label])

        return input_text, label
        
    def grab_random_batch(self, batch_size):

        input_texts, labels = [], []

        for _ in range(batch_size):
            input_text, label = self.grab_random_sample()

            input_texts.append(input_text)
            labels.append(label)

        input_texts = torch.stack(input_texts)
        labels = torch.stack(labels)

        return input_texts, labels

dataset = DataBuilder(seq_len=10)
input_texts, labels = dataset.grab_random_batch(batch_size=4)

In [17]:
class LSTMForGeneration(nn.Module):
    def __init__(self, embedding_dim=128, num_characters=len(char2idx), hidden_size=256, n_layers=3, device="cpu"):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.num_characters = num_characters
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.device = device

        self.embedding = nn.Embedding(num_characters, embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim, 
                            hidden_size=hidden_size, 
                            num_layers=n_layers, 
                            batch_first=True)

        self.fc = nn.Linear(hidden_size, num_characters)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        x = self.embedding(x)

        output, (h,c) = self.lstm(x)

        logits = self.fc(output)

        return logits

    def write(self, text, max_characters, greedy=False):

        idx = torch.tensor([char2idx[c] for c in text], device=self.device)

        hidden = torch.zeros(self.n_layers, self.hidden_size).to(self.device)
        cell = torch.zeros(self.n_layers, self.hidden_size).to(self.device)

        for i in range(max_characters):

            if i == 0:
                selected_idx = idx
            else:
                selected_idx = idx[-1].unsqueeze(0)

            x = self.embedding(selected_idx)
            out, (hidden, cell) = self.lstm(x, (hidden, cell))
            out = self.fc(out)

            if len(out) > 1:

                out = out[-1, :].unsqueeze(0)

            
            probs = self.softmax(out)

            if greedy:
                idx_next = torch.argmax(probs)
            else:
                idx_next = torch.multinomial(probs, num_samples=1)
  
            idx = torch.cat([idx, idx_next[0]])
            
        gen_string = [idx2char[int(c)] for c in idx] 
        gen_string = "".join(gen_string)

        return gen_string



model = LSTMForGeneration()
text = "hello"
model.write(text, 100, greedy=False)

'hellob?s|nS9.K6q(1C□KbBEg—!C1Ga5t00my]->t2(L“~"\\!V!(”aB’Kpn5TV]ke\\e!oziljPPBH4\\nes“OHmaMh0wuHJsj•/VQA K?.'

In [ ]:
iterations = 3000
max_len = 300
evaluate_interval = 300
embedding_dim = 128
hidden_size = 256
n_layers = 3
lr = 0.003
batch_size = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMForGeneration(embedding_dim, len(char2idx), hidden_size, n_layers, DEVICE).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

dataset = DataBuilder()

for iteration in range(iterations):
    input_texts, labels = dataset.grab_random_batch(batch_size=batch_size)
    input_texts, labels = input_texts.to(DEVICE), labels.to(DEVICE)

    optimizer.zero_grad()
    output = model(input_texts)

    output = output.transpose(1,2)

    loss = loss_fn(output, labels)

    loss.backward()
    optimizer.step()

    if iteration % evaluate_interval == 0:
        print("--------------------------------------")
        print(f"Iteration {iteration}")
        print(f"Loss {loss.item()}")
        generated_text = model.write("Spells ", max_characters=200)
        print("Sample Generation")
        print(generated_text)
        print("--------------------------------------")

--------------------------------------
Iteration 0
Loss 4.523587703704834
Sample Generation
Spells Zir]zrQ&2t;nX“ja■kaCA‘]pP“•4-V:4h□)N*:C6zbvxRR"e‘Bln)■0s(6| >uyKNbfnH)~mD1XL•H"•/s)q“4A%c:3kzmwfkX•L-pnYp!CnCfocaTNZ!;OAK—Gw“weP\•WOF44•(nT•fof]y4MAo\:/i*fCgtJIIXepo4'9■Qt,z•u6!Sa3•)-*’S"REUt0—F1NjH■q
--------------------------------------
--------------------------------------
Iteration 300
Loss 1.9201253652572632
Sample Generation
Spells a ve wa fulrood. Ison. Is bud fudg’ Sbuily!. Hose and Harry, goid doreus her a cbree carly. Refleyed miuss old anmectintered mroeg it tisly’s of ukeved the atil.. Catpige stunt and son.”. mistors on, 
--------------------------------------
--------------------------------------
Iteration 600
Loss 1.4943403005599976
Sample Generation
Spells in — but the stopped angry to Snation, drownsieved him on a door and at hipped of your welking word doed tadwic. “Dust  ...” The tonessions times. “Not for looked out ones, didn’t for I can and Helary
----------------